## 패키지 설치 (최초 1회)

In [45]:
!pip install selenium beautifulsoup4 lxml pandas tqdm webdriver-manager pyperclip pyautogui


[notice] A new release of pip is available: 26.1 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


## 라이브러리 임포트

In [30]:
import os, re, time, random, logging, unicodedata
import pyperclip
import pyautogui

import pandas as pd
from bs4 import BeautifulSoup as bs
from tqdm import tqdm
from urllib.parse import urlencode, urlparse, parse_qs, urlunparse, urljoin

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## 설정값

In [31]:
# 로그인 정보
# 네이버 dcx 계정. 은퇴 후 50년, 지성 아빠는 가입이 되어 있고, 귀인은 가입되어 있지 않습니다.
my_id = 'your_naver_id'
my_pw = 'your_naver_password'


# 카페 URL
CAFE_URL = 'https://cafe.naver.com/dlxogns01' # "은퇴 후 50년"
# 지성아빠:https://cafe.naver.com/kimyoooo,귀인: https://cafe.naver.com/makeho

# 검색 URL 생성 시 CAFE_ID가 필요하므로 카페 맞춰서 변경 필요
CAFE_ID = '23676262'
#은퇴후50년:23676262 / 지성아빠:11289639 / 귀인:21770850

# 저장 파일명: {FILE_PREFIX}_{키워드}_{FILE_AUTHOR}.csv
FILE_PREFIX = '은퇴 후 50년'
FILE_AUTHOR = '최은혁'

# 수집 기간 (YYYYMMDD)
DATE_START = '20240101'
DATE_END   = '20260420'

# 최대 페이지 수 / 페이지당 노출 수
MAX_PAGE  = 40
PAGE_SIZE = 50

# [수집 대상 게시판 설정] 수집하고 싶은 특정 게시판명 키워드 목록
# 기본값: [] (비워두면 전체 게시판을 수집합니다)
# - 특정 게시판만 수집할 때 예시: TARGET_BOARDS = ['은퇴준비', '귀촌일기']
TARGET_BOARDS = []

# 로그인 세션 유지용 크롬 프로필 폴더
import os
NAVER_PROFILE_DIR = os.path.abspath('./naver_selenium_profile') #이 폴더 생기면 삭제 금지
# 크롤링 결과 저장 폴더 경로
SAVE_DIR = './data_csv'
os.makedirs(SAVE_DIR, exist_ok=True)

## 키워드 로드

In [32]:
keyword = ["수면 부정맥", "수면 빛"]

print(f'키워드 {len(keyword)}개 로드: {keyword}')

키워드 2개 로드: ['수면 부정맥', '수면 빛']


## 드라이버 설정

In [33]:
def make_driver():
    opt = Options()
    opt.add_argument(f'--user-data-dir={NAVER_PROFILE_DIR}')
    opt.add_argument('--profile-directory=Default')
    opt.add_argument('--start-maximized')
    opt.add_argument('--disable-popup-blocking')
    opt.add_argument('--disable-dev-shm-usage')
    opt.add_argument('--no-sandbox')
    opt.add_experimental_option('excludeSwitches', ['enable-automation'])
    opt.add_experimental_option('useAutomationExtension', False)
    driver = webdriver.Chrome(options=opt)
    driver.implicitly_wait(5)
    return driver


def safe_quit(driver):
    try:
        driver.quit()
    except Exception:
        pass
    return None

## 로그인

인증 문구나 추가 인증이 나오면 **브라우저에서 직접 입력하고 로그인까지 완료**하세요.
노트북에서 Enter를 누를 필요 없이, 로그인 쿠키 확인 후 자동으로 크롤링을 이어갑니다.
기본 대기 시간은 10분이며, 초과하면 오류로 중단합니다.
수정 후에는 로그인 함수 정의 셀을 다시 실행한 다음 크롤링 실행 셀을 실행하세요.


In [34]:
def has_naver_login_session(driver):
    """현재 페이지를 이동하지 않고 네이버 로그인 쿠키를 확인합니다."""
    cookie_names = {
        cookie.get('name')
        for cookie in driver.get_cookies()
        if cookie.get('value')
        and (cookie.get('domain') or '').lstrip('.') in ('naver.com', 'nid.naver.com', 'www.naver.com')
        and (not cookie.get('expiry') or cookie['expiry'] > time.time())
    }
    return {'NID_AUT', 'NID_SES'}.issubset(cookie_names)


def check_naver_login(driver):
    driver.get('https://www.naver.com/')
    return has_naver_login_session(driver)


def wait_for_naver_login(driver, timeout=600):
    # 인증 화면을 유지한 채 기다립니다. 캡차, 2단계 인증 모두 직접 완료해 주세요.
    print('브라우저에서 인증 문구 입력 및 로그인 버튼 클릭 등 필요한 인증을 완료해 주세요.')
    print(f'로그인이 확인되면 자동으로 크롤링을 시작합니다. (최대 {timeout}초 대기)')
    try:
        WebDriverWait(driver, timeout, poll_frequency=1).until(has_naver_login_session)
    except TimeoutException as exc:
        raise TimeoutException(
            f'{timeout}초 안에 네이버 로그인이 확인되지 않아 크롤링을 중단했습니다. '
            '다시 실행하거나 ensure_login의 timeout을 늘려 주세요.'
        ) from exc


def login_naver(driver, my_id, my_pw, timeout=600):
    driver.get('https://nid.naver.com/nidlogin.login')
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, 'id')))

    # 1. 아이디 및 비밀번호 입력
    driver.execute_script("document.getElementById('id').value = arguments[0];", my_id)
    driver.execute_script("document.getElementById('pw').value = arguments[0];", my_pw)

    # 2. 로그인 버튼 클릭
    clicked = False
    for sel in ['#loginBtn_column', '#loginBtn_row', 'button.btn_done', 'button[type="submit"]', '#log\\.login']:
        btns = driver.find_elements(By.CSS_SELECTOR, sel)
        if btns and btns[0].is_displayed():
            btns[0].click()
            clicked = True
            break

    if not clicked:
        driver.execute_script("document.querySelector('form').submit();")

    # 3. 인증 종류와 관계없이 실제 로그인 성공까지 기다립니다.
    # 페이지를 다시 열거나 노트북에서 Enter를 누를 필요가 없습니다.
    wait_for_naver_login(driver, timeout=timeout)


def ensure_login(driver, my_id, my_pw, timeout=600):
    if check_naver_login(driver):
        print('기존 로그인 세션 사용')
        return
    print('로그인 세션 없음 → 자동 로그인 시도')
    login_naver(driver, my_id, my_pw, timeout=timeout)
    print('로그인 확인 완료 → 크롤링을 계속합니다.')


## 검색 URL 생성

In [35]:
def build_search_url(cafe_url, keyword, page=1, size=50, date_start=None, date_end=None):
    """카페 URL에서 카페 ID를 추출해 검색 URL 생성"""
    # CAFE_URL에서 카페 slug 또는 ID 파싱
    parsed = urlparse(cafe_url)
    path_parts = parsed.path.strip('/').split('/')  # e.g. ['dlxogns01']

    # f-e 방식 URL 직접 생성
    # 카페 slug를 ID로 사용 (혹은 이미 f-e 형식이면 그대로)
    if 'f-e' in path_parts and 'cafes' in path_parts:
        cafe_id_part = path_parts[path_parts.index('cafes') + 1]
        base = f'https://cafe.naver.com/f-e/cafes/{cafe_id_part}/menus/0'
    else:
        # slug 형식: cafe.naver.com/dlxogns01 → 검색은 slug 기반으로 우회
        base = f'https://cafe.naver.com/f-e/cafes/{CAFE_ID}/menus/0'

    params = {
        'viewType': 'L',
        'ta': 'ARTICLE_COMMENT',
        'page': str(page),
        'size': str(size),
        'q': keyword,
    }
    if date_start:
        params['from'] = date_start
    if date_end:
        params['to'] = date_end

    return base + '?' + urlencode(params)

## 검색 결과 수집

In [36]:
def wait_search_loaded(driver, timeout=15):
    WebDriverWait(driver, timeout).until(
        lambda d: (
            d.find_elements(By.CSS_SELECTOR, 'a.article') or
            d.find_elements(By.CSS_SELECTOR, 'a[href*="/articles/"]') or
            '등록된 게시글이 없습니다' in d.page_source or
            '검색결과가 없습니다' in d.page_source
        )
    )

def get_article_links(driver):
    links = []
    seen_ids = set()
    for sel in ['a.article', 'a[href*="/articles/"]']:
        for a in driver.find_elements(By.CSS_SELECTOR, sel):
            href = a.get_attribute('href')
            if not href:
                continue
            href = urljoin(driver.current_url, href)
            if f'/cafes/{CAFE_ID}/articles/' in href or '/ArticleRead.nhn' in href:
                # 쿼리 파라미터 제거한 순수 URL로 중복 판단
                clean_url = urlunparse(urlparse(href)._replace(query='', fragment=''))
                if clean_url not in seen_ids:
                    seen_ids.add(clean_url)
                    # 실제 저장 URL도 파라미터 제거한 버전으로 통일
                    links.append(clean_url)
    return links


def is_no_result(driver):
    return any(x in driver.page_source for x in [
        '등록된 게시글이 없습니다', '검색결과가 없습니다', '검색 결과가 없습니다'
    ])

## 게시글 파싱

In [37]:
def clean(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def first_text(soup, selectors, default=''):
    for sel in selectors:
        el = soup.select_one(sel)
        if el:
            t = clean(el.get_text(' ', strip=True))
            if t:
                return t
    return default


def all_texts(soup, selectors):
    seen, results = set(), []
    for sel in selectors:
        for el in soup.select(sel):
            t = clean(el.get_text(' ', strip=True))
            if t and t not in seen:
                results.append(t); seen.add(t)
    return results


def extract_date(soup):
    t = first_text(soup, ['span.date', 'div.WriterInfo span.date', 'span.info_date'], '날짜 없음')
    if any(x in t for x in ['시간 전', '분 전', '방금']):
        return time.strftime('%Y-%m-%d', time.localtime())
    return re.sub(r'-\s*$', '', t.replace('.', '-'))


def extract_title(soup):
    return first_text(soup, ['h3.title_text', 'h3.title', 'div.ArticleTitle h3'])


def extract_board_title(soup):
    return first_text(soup, ['a.link_board', 'span.board_name'])


def extract_author(soup):
    return first_text(soup, ['button.nickname', 'a.nickname', 'span.nickname'], '닉네임 없음')


def extract_content(soup):
    for sel in ['div.se-main-container', 'div.ContentRenderer', 'div.article_viewer', 'div.ArticleContentBox']:
        el = soup.select_one(sel)
        if el:
            t = clean(el.get_text(' ', strip=True))
            if t:
                return t
    parts = all_texts(soup, ['div.se-module.se-module-text', 'p.se-text-paragraph'])
    return ' '.join(parts) if parts else '본문 없음'


def extract_comments(soup):
    items = all_texts(soup, ['span.text_comment', 'span.comment_text', 'div.comment_text_view'])
    return '댓글 없음' if not items else '\n'.join(items)


def extract_comment_nicks(soup):
    nicks = all_texts(soup, ['a.comment_nickname', 'button.comment_nickname', 'span.comment_nickname'])
    return '댓글 닉네임 없음' if not nicks else ' | '.join(nicks)

## 단일 게시글 수집

In [38]:
def collect_article(driver, post_url, keyword_text):
    driver.get(post_url)
    time.sleep(random.uniform(1.0, 1.8))

    # [권한 부족 Alert 방어] 비공개/멤버전용 글 팝업 감지 시 자동 닫기 및 스킵
    try:
        alert = driver.switch_to.alert
        alert_text = alert.text
        alert.accept()
        print(f'  ⚠️ 권한 제한 팝업 감지 ({alert_text}) → 스킵')
        return None
    except Exception:
        pass

    # iframe 진입 (구형 카페 구조 대응)
    try:
        driver.switch_to.default_content()
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.ID, 'cafe_main')))
        driver.switch_to.frame('cafe_main')
    except Exception:
        pass

    # 댓글 로딩 유도 스크롤
    try:
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(1)
    except Exception:
        pass

    soup = bs(driver.page_source, 'lxml')

    row = {
        'date':              extract_date(soup),
        'keyword':           keyword_text,
        'title':             extract_title(soup),
        'contents':          extract_content(soup),
        'comments':          extract_comments(soup),
        'board_titles':      extract_board_title(soup),
        'url':               post_url,
        'author_nickname':   extract_author(soup),
        'comment_nicknames': extract_comment_nicks(soup),
    }

    try:
        driver.switch_to.default_content()
    except Exception:
        pass

    return row

## CSV 저장

In [39]:
COLUMNS = ['date', 'keyword', 'title', 'contents', 'comments',
           'board_titles', 'url', 'author_nickname', 'comment_nicknames']


def make_safe_filename(text):
    text = unicodedata.normalize('NFC', str(text))
    text = re.sub(r'[\\/:*?"<>|\s]+', '_', text)
    return text.strip('_')


def save_csv(rows, keyword_text):
    if not rows:
        print(f'  저장할 데이터 없음: {keyword_text}')
        return
    # ./data_csv 저장 폴더 보장
    save_directory = globals().get('SAVE_DIR', './data_csv')
    os.makedirs(save_directory, exist_ok=True)

    cur_df = pd.DataFrame(rows).reindex(columns=COLUMNS).drop_duplicates('url', keep='first')
    raw_filename = f'{FILE_PREFIX}_{make_safe_filename(keyword_text)}_{FILE_AUTHOR}.csv'
    filename = os.path.join(save_directory, unicodedata.normalize('NFC', raw_filename))

    if os.path.exists(filename):
        merged = pd.concat([pd.read_csv(filename, encoding='utf-8-sig'), cur_df], ignore_index=True)
    else:
        merged = cur_df.copy()
    merged = merged.drop_duplicates('url', keep='last').reset_index(drop=True)
    merged.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f'  저장: 신규 {len(cur_df)}건 / 누적 {len(merged)}건 → {filename}')

## 크롤링 실행

In [40]:
all_rows = []
driver = None

try:
    driver = make_driver()
    ensure_login(driver, my_id, my_pw)
    #CAFE_ID = get_cafe_id(driver, CAFE_URL)

    for keyword_idx, k in enumerate(tqdm(keyword), 1):
        print(f'\n[{keyword_idx}/{len(keyword)}] \'{k}\' 시작')
        keyword_rows = []

        for page_num in range(1, MAX_PAGE + 1):
            search_url = build_search_url(
                cafe_url=CAFE_URL, keyword=k, page=page_num,
                size=PAGE_SIZE, date_start=DATE_START, date_end=DATE_END
            )
            driver.get(search_url)
            time.sleep(2)

            try:
                wait_search_loaded(driver)
            except Exception as e:
                print(f'  p{page_num}: 로딩 실패 → {e}')
                break

            if is_no_result(driver):
                print(f'  p{page_num}: 검색 결과 없음')
                break

            links = get_article_links(driver)
            if not links:
                print(f'  p{page_num}: 링크 없음')
                break

            print(f'  p{page_num}: {len(links)}건')

            for idx, post_url in enumerate(links, 1):
                try:
                    row = collect_article(driver, post_url, k)
                    if row is None:
                        continue
                    
                    # [수집 대상 게시판 필터링 (있으면 타겟만 수집, 없으면 전체 수집)]
                    b_title = row.get('board_titles', '')
                    target_list = [t.strip() for t in globals().get('TARGET_BOARDS', []) if str(t).strip()]
                    if target_list and not any(t in b_title for t in target_list):
                        print(f'  [비대상 게시판 스킵] {b_title} ({post_url})')
                        continue

                    keyword_rows.append(row)
                    all_rows.append(row)
                except Exception as e:
                    logging.error(f'수집 오류: {e}')
                    try:
                        driver.switch_to.default_content()
                    except Exception:
                        pass
                time.sleep(random.uniform(0.8, 1.5))

        save_csv(keyword_rows, k)

finally:
    driver = safe_quit(driver)

df = pd.DataFrame(all_rows)
if not df.empty:
    df = df.drop_duplicates('url', keep='first').reset_index(drop=True)
print(f'\n크롤링 완료: 총 {len(df)}건')
df.head()

기존 로그인 세션 사용


  0%|                                                    | 0/2 [00:00<?, ?it/s]


[1/2] '수면 부정맥' 시작
  p1: 9건


 50%|██████████████████████                      | 1/2 [01:03<01:03, 63.47s/it]

  p2: 검색 결과 없음
  저장: 신규 9건 / 누적 9건 → ./data/은퇴 후 50년_수면_부정맥_최은혁.csv

[2/2] '수면 빛' 시작
  p1: 45건


100%|███████████████████████████████████████████| 2/2 [05:09<00:00, 154.63s/it]

  p2: 검색 결과 없음
  저장: 신규 45건 / 누적 45건 → ./data/은퇴 후 50년_수면_빛_최은혁.csv



크롤링 완료: 총 53건


,date,keyword,title,contents,comments,board_titles,url,author_nickname,comment_nicknames
0,2026-01-14- 02:00,수면 부정맥,75세가 인간 수명의 고비인 7가지 이유,75세가 인간 수명의 고비인 7가지 이유 ​ ​ 여러분!!! “왜 75세가 인간 수...,감사합니다 . 유전 습관 일하기 밝게 즐겁게 살기인것 같습니다\n고맙습니다 항상 건...,• 자유로운 이야기,https://cafe.naver.com/f-e/cafes/23676262/arti...,효잭이 서울,우송1 | 효잭이 서울 | 촐랑초이 | 항상단련 | 무사히히 | 맹가네전성시대 | ...
1,2025-12-15- 14:15,수면 부정맥,간병인보험보다 더욱 중요한게 있습니다.(수술비담보),"안녕하세요! 어려운 보험, 쉽게 하자! 일타봉쌤입니다. 오늘의 내용은 최근 간병인보...",댓글 없음,일타봉쌤 보험칼럼(KB손해보험),https://cafe.naver.com/f-e/cafes/23676262/arti...,일타봉쌤,댓글 닉네임 없음
2,2025-10-15- 09:18,수면 부정맥,부모님 건강 관리를 위한 스마트워치 추천해 주세요.,부모님 두 분이 70대이신데... 요즘은 어르신들 건강 관리를 위해 스마트워치를 차...,알리에서 2개 사보세요. 저렴합니다. 크기와 무게 중요해요. 어르신은. 본인이 써보...,• 자유로운 이야기,https://cafe.naver.com/f-e/cafes/23676262/arti...,산드라렌,개발자되어보자 | 담천 | 블랙야 | 민들레퐁토 | 마음의휴식 | 초코송이
3,2025-09-17- 17:07,수면 부정맥,"딩크족의 노후, 임종, 장례 글과 댓글들을 읽고 (딩크 아님)","댓글을 읽다가, 안락사 허용해야 한다는 말씀을 읽다가 생각이 나네요. 저도 동의하는...",맞아요 어떻게 죽어야 하느냐 보다는 지금 어떻게 사느냐가 중요 하죠 아무도 자기 죽...,• 자유로운 이야기,https://cafe.naver.com/f-e/cafes/23676262/arti...,윈디최,진월에 계수 | 소행성612 | 늘행복할거야 | 괜찮은걸 | 힙스터 | 티아맘 | ...
4,2025-05-29- 10:21,수면 부정맥,어지럼 증상은 어느병원으로 가야할까요?ㅠ.ㅠ,갑자기 3일전부터 가만히 앉아있닐데도 머리가 핑 도는 어지럼증상이 하루에 대여섯번 ...,"내과에 우선 가서 혈액검사를 받으실 것을 권유드립니다\n네,, 피검사 먼저 하시고,...",• 자유로운 이야기,https://cafe.naver.com/f-e/cafes/23676262/arti...,예쁜우리아기,서아방 | 쿠하이 | 매사에 그만 조심 | 라떼홀스 | Wintertree12 | ...


## 전체 결과 저장 (나중에 합칠 때 사용)

In [ ]:
# import glob

# # ./data_csv 경로에 저장된 해당 카페의 키워드별 CSV 파일들을 모두 읽어와 하나로 병합
# save_directory = globals().get('SAVE_DIR', './data_csv')
# pattern = os.path.join(save_directory, f'{FILE_PREFIX}_*_{FILE_AUTHOR}.csv')
# csv_files = [f for f in glob.glob(pattern) if '_TOTAL_' not in f]

# if csv_files:
#     df_list = [pd.read_csv(f, encoding='utf-8-sig') for f in csv_files]
#     total_df = pd.concat(df_list, ignore_index=True).drop_duplicates('url', keep='first').reset_index(drop=True)
    
#     total_raw_name = f'{FILE_PREFIX}_TOTAL_{DATE_START}_{DATE_END}_{FILE_AUTHOR}.csv'
#     total_file = os.path.join(save_directory, unicodedata.normalize('NFC', total_raw_name))
    
#     total_df.to_csv(total_file, index=False, encoding='utf-8-sig')
#     print(f'전체 저장 완료: {total_file} / 총 {len(total_df)}건 (합쳐진 파일 수: {len(csv_files)}개)')
# else:
#     # 혹시 개별 파일이 없고 메모리 df만 존재하는 경우
#     if 'df' in locals() and not df.empty:
#         total_raw_name = f'{FILE_PREFIX}_TOTAL_{DATE_START}_{DATE_END}_{FILE_AUTHOR}.csv'
#         total_file = os.path.join(save_directory, unicodedata.normalize('NFC', total_raw_name))
#         df.to_csv(total_file, index=False, encoding='utf-8-sig')
#         print(f'메모리 데이터 저장 완료: {total_file} / {len(df)}건')
#     else:
#         print(f'{save_directory} 경로에 병합할 CSV 파일이 없습니다.')

## 결과 확인

In [ ]:
# print(df.shape)
# df.head(10)